# Representative Shapes by Embedding Region

For each embedding (UMAP, PHATE), for each modality (cell, nucleus): k-means
clusters the full set of embedded cells (every subclass, every available
frame - the same cells `01_subclasses_over_time.ipynb` plots) into a small
number of regions, then picks the one real cell closest to each region's
own centroid and shows its actual shape, cropped from its extracted mask.

This runs once per embedding per modality, not once per notebook - the
embedding space itself doesn't change between notebooks, only which
subset of it they each plot.

Requires `main.py` to have been run first.


In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/04_vem_full_timecourse/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR/MASKS_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)

from shared.load_data import load_full_dataset
from shared.representative_shapes import export_representative_shapes


## Plot: embedding regions and their representative shapes

For each embedding method, one PDF and one SVG (in its own `svgs/`
subfolder), written to `output/plots/00_representative_shapes/`.


In [ ]:
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '00_representative_shapes')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# how many regions to cluster the embedding into, and how large a crop
# (half-width in pixels) to pull each representative cell's shape from -
# half_width is a guess at "large enough for any cell in this dataset";
# crop_cell_shape prints a clear warning if a representative cell's own
# shape doesn't fully fit, so bump this up if that happens
N_REGIONS   = 5
HALF_WIDTH  = 150

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    For each embedding method, load this modality's full embedding (every
    subclass, every available frame - no filtering), cluster it into
    N_REGIONS regions, and export each region's representative shape.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        dfs: {embedding_method: the full merged dataset for this
              modality and embedding method, plus a 'region' column}
              (in-memory only)
    '''
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} cells across {df['frame'].nunique()} frames")

        df = export_representative_shapes(
            df, config.MASKS_DIR, modality, N_REGIONS, embedding_method.upper(),
            f'{MODALITY_NAMES[modality]} morphology (representative regions)',
            NOTEBOOK_PLOTS_DIR, f'{modality}_{embedding_method}', half_width=HALF_WIDTH,
        )
        dfs[embedding_method] = df

    return dfs


## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc')
